# Improved Sequence Pattern Extraction

## Motivation

Smart-home automation is not solely about learning *when* devices operate.
Many household behaviours are defined by **ordered sequences of actions** rather
than individual events.

Examples include:

- Leaving for work
    - Bedroom Presence → LEAVE
    - Bedroom Fan → OFF
    - Bedroom Light → OFF
    - Main Door → LOCK

- Going to sleep
    - Living Room TV → OFF
    - Bedroom Light → ON
    - Bedroom Presence → ENTER

These routines are composed of multiple events occurring close together in time.

---

## Limitations of the Original Algorithm

The original implementation extracted sequences by requiring **exact signature
matching**.

For example,

Day 1

LEAVE
↓
FAN OFF
↓
LIGHT OFF

Day 2

LEAVE
↓
LIGHT OFF
↓
FAN OFF

would be considered two completely different routines despite representing the
same household behaviour.

Similarly,

LEAVE
↓
FAN OFF
↓
LIGHT OFF

and

LEAVE
↓
LIGHT OFF

would also be considered different patterns.

Real households rarely execute routines identically every day.

---

## Goals of the Improved Algorithm

The improved extractor should:

- Discover recurring routines even when minor variations exist.
- Allow optional or missing actions.
- Compare routines based on similarity rather than exact equality.
- Count unique days instead of duplicate sessions.
- Learn typical execution time.
- Produce confidence scores that reflect:
    - frequency,
    - timing consistency,
    - sequence completeness.

The final output should remain deterministic, explainable, and computationally
lightweight, making it suitable for embedded smart-home intelligence systems.

# Overall Pipeline

The improved sequence extraction pipeline is:

Historical Event Logs

        ↓

Chronological Sorting

        ↓

Adaptive Sessionization

        ↓

Canonical Session Representation

        ↓

Pairwise Sequence Similarity (LCS)

        ↓

Cluster Similar Sessions

        ↓

Representative Sequence

        ↓

Calculate

• Support
• Circular Mean Start Time
• Circular Standard Deviation
• Completeness Score

        ↓

Confidence Score

        ↓

Final Sequence Pattern

In [5]:
from collections import defaultdict
from datetime import datetime
import statistics
import math

from itertools import combinations

## Required Libraries

The implementation intentionally relies almost entirely on Python's standard
library.

Unlike deep-learning approaches, no heavy machine-learning model is required.

This keeps the algorithm:

- deterministic,
- interpretable,
- lightweight,
- easy to debug.

## Example Dataset

Unlike the previous notebook, this dataset intentionally includes
imperfections.

Examples include:

- different ordering of actions,
- optional actions,
- missing actions,
- unrelated noise.

A robust sequence mining algorithm should still recognize these as the same
underlying household routine.

In [6]:
# example data set

events = [

# --------------------------
# Day 1
# --------------------------

("bedroom_presence","LEAVE","2026-06-01 08:00"),
("bedroom_fan","OFF","2026-06-01 08:02"),
("bedroom_light","OFF","2026-06-01 08:04"),
("main_door","LOCK","2026-06-01 08:06"),

# --------------------------
# Day 2
# --------------------------

("bedroom_presence","LEAVE","2026-06-02 08:01"),
("bedroom_light","OFF","2026-06-02 08:03"),
("bedroom_fan","OFF","2026-06-02 08:04"),
("main_door","LOCK","2026-06-02 08:07"),

# --------------------------
# Day 3
# Missing Door Lock
# --------------------------

("bedroom_presence","LEAVE","2026-06-03 07:59"),
("bedroom_fan","OFF","2026-06-03 08:01"),
("bedroom_light","OFF","2026-06-03 08:04"),

# --------------------------
# Day 4
# Extra Kitchen Event
# --------------------------

("bedroom_presence","LEAVE","2026-06-04 08:02"),
("kitchen_light","OFF","2026-06-04 08:03"),
("bedroom_fan","OFF","2026-06-04 08:04"),
("bedroom_light","OFF","2026-06-04 08:05"),
("main_door","LOCK","2026-06-04 08:07"),

# Random unrelated activity

("tv","ON","2026-06-04 20:15"),
("kitchen_light","ON","2026-06-04 13:30")

]

## Event Parsing

Raw string timestamps are converted into Python datetime objects.

This enables chronological sorting and accurate time difference calculations.

In [7]:
parsed_events=[]

for device,action,ts in events:

    parsed_events.append({

        "device":device,

        "action":action,

        "timestamp":datetime.strptime(
            ts,
            "%Y-%m-%d %H:%M"
        )

    })

## Step 1 — Chronological Ordering

Sequence mining fundamentally depends on temporal order.

The first step therefore sorts every event by timestamp.

Only after this ordering can the algorithm determine which events belong to the
same routine.

In [8]:
ordered=sorted(
    parsed_events,
    key=lambda e:e["timestamp"]
)

ordered[:5]

[{'device': 'bedroom_presence',
  'action': 'LEAVE',
  'timestamp': datetime.datetime(2026, 6, 1, 8, 0)},
 {'device': 'bedroom_fan',
  'action': 'OFF',
  'timestamp': datetime.datetime(2026, 6, 1, 8, 2)},
 {'device': 'bedroom_light',
  'action': 'OFF',
  'timestamp': datetime.datetime(2026, 6, 1, 8, 4)},
 {'device': 'main_door',
  'action': 'LOCK',
  'timestamp': datetime.datetime(2026, 6, 1, 8, 6)},
 {'device': 'bedroom_presence',
  'action': 'LEAVE',
  'timestamp': datetime.datetime(2026, 6, 2, 8, 1)}]

# Step 2 — Sessionization

A household routine is rarely a single event.

Instead, routines occur as **bursts of activity**.

For example,

08:00 Bedroom Presence → LEAVE

08:02 Bedroom Fan → OFF

08:04 Bedroom Light → OFF

08:06 Main Door → LOCK

These events are clearly related and should be treated as one logical routine.

The purpose of sessionization is to split the entire event timeline into these
independent bursts of activity.

---

### Why not simply compare every event?

Without sessionization, events from completely different contexts could become
mixed together.

Example:

08:00 Bedroom Fan OFF

08:04 Bedroom Light OFF

13:30 Kitchen Light ON

20:15 TV ON

These four events clearly do not belong to one routine.

Sessionization prevents unrelated activities from being grouped together.

## Session Boundary Rule

Two consecutive events belong to the same session if

Time Difference ≤ MAX_GAP_MINUTES

Otherwise,

a new session begins.

Example

08:00 Leave

08:02 Fan OFF

08:05 Light OFF

↓

One session

---------------------------------

08:00 Leave

08:02 Fan OFF

09:00 TV ON

↓

Two sessions

In [9]:
# we will kep the same configurable parameters

MAX_GAP_MINUTES = 10

MIN_SEQUENCE_LENGTH = 2

MAX_SEQUENCE_LENGTH = 8

In [10]:
def sessionize(events):

    sessions = []

    current_session = []

    previous_time = None

    for event in events:

        if previous_time is not None:

            gap = (
                event["timestamp"]
                -
                previous_time
            ).total_seconds() / 60

            if gap > MAX_GAP_MINUTES:

                sessions.append(current_session)

                current_session = []

        current_session.append(event)

        previous_time = event["timestamp"]

    if current_session:

        sessions.append(current_session)

    return sessions

In [11]:
sessions = sessionize(ordered)

len(sessions)

6

## Session Visualization

Each printed block represents one independent household activity.

Example

Session 1

Bedroom Presence LEAVE

Bedroom Fan OFF

Bedroom Light OFF

Main Door LOCK

This session represents a complete departure routine.

Later sessions correspond to subsequent days.

Notice that unrelated events such as

TV ON

Kitchen Light ON

form independent sessions rather than contaminating the departure routine.

In [12]:
for i, session in enumerate(sessions):

    print("="*40)

    print(f"Session {i+1}")

    print("="*40)

    for event in session:

        print(
            event["timestamp"].strftime("%Y-%m-%d %H:%M"),
            "|",
            event["device"],
            "|",
            event["action"]
        )

    print()

Session 1
2026-06-01 08:00 | bedroom_presence | LEAVE
2026-06-01 08:02 | bedroom_fan | OFF
2026-06-01 08:04 | bedroom_light | OFF
2026-06-01 08:06 | main_door | LOCK

Session 2
2026-06-02 08:01 | bedroom_presence | LEAVE
2026-06-02 08:03 | bedroom_light | OFF
2026-06-02 08:04 | bedroom_fan | OFF
2026-06-02 08:07 | main_door | LOCK

Session 3
2026-06-03 07:59 | bedroom_presence | LEAVE
2026-06-03 08:01 | bedroom_fan | OFF
2026-06-03 08:04 | bedroom_light | OFF

Session 4
2026-06-04 08:02 | bedroom_presence | LEAVE
2026-06-04 08:03 | kitchen_light | OFF
2026-06-04 08:04 | bedroom_fan | OFF
2026-06-04 08:05 | bedroom_light | OFF
2026-06-04 08:07 | main_door | LOCK

Session 5
2026-06-04 13:30 | kitchen_light | ON

Session 6
2026-06-04 20:15 | tv | ON



# Improvement 1 — Event Compression

Real smart-home logs often contain repeated actions.

Example

Bedroom Light OFF

Bedroom Light OFF

Bedroom Light OFF

These repeated events usually originate from sensors or duplicate state updates.

Treating all of them as unique steps makes routines appear different even though
the household behaviour has not changed.

Therefore, before constructing sequence signatures, consecutive duplicate
events should be removed.

In [14]:
def compress_session(session):

    compressed = []

    previous_step = None

    for event in session:

        step = (
            event["device"],
            event["action"]
        )

        if step != previous_step:

            compressed.append(event)

        previous_step = step

    return compressed

In [15]:
compressed_sessions = [

    compress_session(session)

    for session in sessions

]

In [16]:
# display the compressed sessions
for i, session in enumerate(compressed_sessions):

    print("="*40)

    print(f"Compressed Session {i+1}")

    print("="*40)

    for event in session:

        print(
            event["device"],
            event["action"]
        )

    print()

Compressed Session 1
bedroom_presence LEAVE
bedroom_fan OFF
bedroom_light OFF
main_door LOCK

Compressed Session 2
bedroom_presence LEAVE
bedroom_light OFF
bedroom_fan OFF
main_door LOCK

Compressed Session 3
bedroom_presence LEAVE
bedroom_fan OFF
bedroom_light OFF

Compressed Session 4
bedroom_presence LEAVE
kitchen_light OFF
bedroom_fan OFF
bedroom_light OFF
main_door LOCK

Compressed Session 5
kitchen_light ON

Compressed Session 6
tv ON



## Why Compression Matters

Suppose a motion sensor sends

Bedroom Light OFF

three consecutive times.

Without compression

LEAVE

LIGHT OFF

LIGHT OFF

LIGHT OFF

LOCK

With compression

LEAVE

LIGHT OFF

LOCK

The compressed representation better captures the actual behaviour while
removing redundant sensor updates.

# Step 3 — Canonical Session Representation

The original implementation represented a routine simply as

device:action

tuples.

We retain this idea because it is compact, deterministic and human-readable.

However, the canonical representation is now built **after compression**, making
it considerably more robust to noisy sensor logs.

In [18]:
def canonical_signature(session):

    signature = []

    for event in session:

        signature.append(

            f"{event['device']}:{event['action']}"

        )

    return tuple(

        signature[:MAX_SEQUENCE_LENGTH]

    )

In [19]:
# generate signatures
signatures = []

for session in compressed_sessions:

    if len(session) < MIN_SEQUENCE_LENGTH:

        continue

    signatures.append(

        canonical_signature(session)

    )

signatures

[('bedroom_presence:LEAVE',
  'bedroom_fan:OFF',
  'bedroom_light:OFF',
  'main_door:LOCK'),
 ('bedroom_presence:LEAVE',
  'bedroom_light:OFF',
  'bedroom_fan:OFF',
  'main_door:LOCK'),
 ('bedroom_presence:LEAVE', 'bedroom_fan:OFF', 'bedroom_light:OFF'),
 ('bedroom_presence:LEAVE',
  'kitchen_light:OFF',
  'bedroom_fan:OFF',
  'bedroom_light:OFF',
  'main_door:LOCK')]

## Canonical Signatures

Example

Bedroom Presence LEAVE

↓

Bedroom Fan OFF

↓

Bedroom Light OFF

↓

Main Door LOCK

becomes

(
"bedroom_presence:LEAVE",

"bedroom_fan:OFF",

"bedroom_light:OFF",

"main_door:LOCK"
)

Unlike the original implementation, these signatures are no longer affected by
duplicate sensor events.

However, they still require exact matching.

The next section removes this final limitation by introducing sequence
similarity using the Longest Common Subsequence (LCS) algorithm.

# Step 4 — Sequence Similarity

The original algorithm compared sessions using **exact equality**.

Example

Session A

LEAVE
↓

FAN OFF
↓

LIGHT OFF
↓

DOOR LOCK

Session B

LEAVE
↓

LIGHT OFF
↓

FAN OFF
↓

DOOR LOCK

Although both sessions clearly represent the same departure routine,
their signatures are different.

Instead of exact matching, we compare two routines by measuring how similar
their ordered actions are.

The similarity measure used here is the **Longest Common Subsequence (LCS)**.

## Longest Common Subsequence (LCS)

LCS measures the longest ordered sequence of actions shared by two routines.

Example

Sequence A

LEAVE
↓

FAN OFF
↓

LIGHT OFF
↓

LOCK

Sequence B

LEAVE
↓

LIGHT OFF
↓

LOCK

LCS

LEAVE
↓

LIGHT OFF
↓

LOCK

Length = 3

The routines are therefore highly similar even though one action is missing.

Unlike exact matching, LCS naturally tolerates

• missing actions

• optional actions

• small ordering variations

In [21]:
# implement lcs

def lcs_length(seq1, seq2):

    m = len(seq1)
    n = len(seq2)

    dp = [

        [0]*(n+1)

        for _ in range(m+1)

    ]

    for i in range(1,m+1):

        for j in range(1,n+1):

            if seq1[i-1] == seq2[j-1]:

                dp[i][j] = dp[i-1][j-1] + 1

            else:

                dp[i][j] = max(

                    dp[i-1][j],

                    dp[i][j-1]

                )

    return dp[m][n]

The dynamic programming table computes the LCS efficiently.

Time Complexity

O(m × n)

where

m = length of first sequence

n = length of second sequence

Since household routines usually contain fewer than 10 actions,
this computation is extremely fast.

In [22]:
def sequence_similarity(seq1, seq2):

    lcs = lcs_length(

        seq1,

        seq2

    )

    return lcs / max(

        len(seq1),

        len(seq2)

    )

In [23]:
for i,s1 in enumerate(signatures):

    for j,s2 in enumerate(signatures):

        if i < j:

            print(

                f"{i}-{j}",

                round(

                    sequence_similarity(

                        s1,

                        s2

                    ),

                    3

                )

            )

0-1 0.75
0-2 0.75
0-3 0.8
1-2 0.5
1-3 0.6
2-3 0.6


## Similarity Score

The similarity score ranges between

0 and 1

1

identical routines

0.75

highly similar routines

0

completely unrelated routines

This score forms the basis for grouping routines together.

In [25]:
matrix = []

for s1 in signatures:

    row = []

    for s2 in signatures:

        row.append(

            round(

                sequence_similarity(

                    s1,

                    s2

                ),

                2

            )

        )

    matrix.append(row)

matrix

[[1.0, 0.75, 0.75, 0.8],
 [0.75, 1.0, 0.5, 0.6],
 [0.75, 0.5, 1.0, 0.6],
 [0.8, 0.6, 0.6, 1.0]]

## Similarity Matrix

Each entry

(i,j)

represents how similar two sessions are.

Example

        S1   S2   S3   S4

S1     1.0  .80  .75  .90

S2     .80 1.0   .72  .84

S3     .75 .72   1.0  .68

S4     .90 .84   .68  1.0

Sessions with high similarity likely belong to the same underlying
household routine.

# Step 5 — Group Similar Sessions

Instead of counting identical signatures,

we cluster together sessions whose similarity exceeds a threshold.

Example

Similarity

0.90

↓

Same Routine

Similarity

0.35

↓

Different Routine

This allows the algorithm to learn routines despite small variations.

In [26]:
SIMILARITY_THRESHOLD = 0.75

clusters = []

visited = set()

for i, seq in enumerate(signatures):

    if i in visited:

        continue

    cluster = [i]

    visited.add(i)

    for j in range(i+1,len(signatures)):

        score = sequence_similarity(

            seq,

            signatures[j]

        )

        if score >= SIMILARITY_THRESHOLD:

            cluster.append(j)

            visited.add(j)

    clusters.append(cluster)

In [27]:
for i,cluster in enumerate(clusters):

    print()

    print(

        f"Cluster {i+1}"

    )

    print("-"*30)

    for idx in cluster:

        print(

            signatures[idx]

        )


Cluster 1
------------------------------
('bedroom_presence:LEAVE', 'bedroom_fan:OFF', 'bedroom_light:OFF', 'main_door:LOCK')
('bedroom_presence:LEAVE', 'bedroom_light:OFF', 'bedroom_fan:OFF', 'main_door:LOCK')
('bedroom_presence:LEAVE', 'bedroom_fan:OFF', 'bedroom_light:OFF')
('bedroom_presence:LEAVE', 'kitchen_light:OFF', 'bedroom_fan:OFF', 'bedroom_light:OFF', 'main_door:LOCK')


## Why Clustering?

Suppose four departure routines were observed.

Routine 1

LEAVE

↓

FAN OFF

↓

LIGHT OFF

↓

LOCK

Routine 2

LEAVE

↓

LIGHT OFF

↓

LOCK

Routine 3

LEAVE

↓

FAN OFF

↓

LIGHT OFF

↓

LOCK

Routine 4

LEAVE

↓

LIGHT OFF

↓

FAN OFF

↓

LOCK

The original algorithm would generate

three different patterns.

The improved algorithm clusters these sessions into one representative
routine because they are sufficiently similar.

At this stage the algorithm has completed the most difficult task.

Instead of requiring identical routines,

it has grouped together sessions representing the same household behaviour.

The remaining stages will now compute

• representative sequence

• support

• timing consistency

• completeness

• confidence

before producing the final sequence patterns.